***

Preparing Workspace

***

In [ ]:


## Importing packages ---

import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import ipywidgets as widgets

pd.options.display.float_format = '{:.1f}'.format


## Setting file paths ---

user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

if user == 'jfontes':

    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    path_out  = os.path.join(path_users
                             , 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
                             , 'Process Revamp'
                             , 'Task 9. Collect new data'
                             , 'Census')

path_code    = os.path.join(path_git, 'Data', 'EIA')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')


## User defined functions ---

exec(open(os.path.join(path_config0, 'Functions.py')).read())


## Setting API key ---

# Obtain API Key from the following source 
# https://www.eia.gov/opendata/documentation.php
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()



***

Route 1

***

In [ ]:
root_ = 'https://api.eia.gov/v2'
api_key_ = f'/?api_key={api_key}'


has_routes = ['coal', 'electricity', 'natural-gas', 'nuclear-outages', 'petroleum', 'densified-biomass', 'aeo', 'co2-emissions']
categories_to_keep = ['coal', 'electricity', 'natural-gas', 'nuclear-outages', 'petroleum', 'densified-biomass', 'co2-emissions']

list_df_cat = []

for cat in tqdm(categories_to_keep):

    cat_ = f'/{cat}'
    query = f"{root_}{cat_}{api_key_}"
    
    response = requests.get(query).text
    response = response.replace('null', '"null"')
    response = ast.literal_eval(response)
    
    
    list_df_id = []

    if cat in has_routes:
        for id in range(len(response['response']['routes'])):
            df_routes = pd.DataFrame(response['response']['routes'][id], index = [0])
            df_routes = df_routes.rename(columns = {'id':'route1', 'name':'route1_name'})
            list_df_id.append(df_routes)
    
    
    df_routes = pd.concat(list_df_id)
    df_routes = df_routes.reset_index(drop = True)
    
    
    df_routes['category'] = cat
    list_df_cat.append(df_routes)


df_routes = pd.concat(list_df_cat)
df_routes = df_routes.reset_index(drop = True)
df_routes = df_routes.set_index('category').reset_index()
df_routes['route1_name'] = df_routes['route1_name'].str.replace('\\/', '/')
df_routes['route1_name'] = df_routes['route1_name'].str.replace('\\' , '/')
df_routes['route1_name'] = df_routes['route1_name'].str.replace(' / ', '/')
df_routes.columns = ['category', 'route1', 'route1_name', 'description']

display(df_routes.head(), df_routes.tail())



***

Route 2

***

In [ ]:
categories = list(df_routes.category.unique())
categories

In [ ]:
root_ = 'https://api.eia.gov/v2'


list_df_cat = []

df_loop = df_routes.copy()

for cat in categories:

    print(cat)
    cat_ = f'/{cat}'
    
    routes = list(df_loop[df_loop['category'] == cat].route1.unique())
    print(routes)
    
    list_df_routes = []
    
    for route in routes:
    
        try:
            route_ = f'/{route}'
            api_key_ = f'/?api_key={api_key}'
            
            query = f"{root_}{cat_}{route_}{api_key_}"
            
            print(query)    
            
            response = requests.get(query).text
            response = response.replace('null', '"null"')
            response = ast.literal_eval(response)
            
            df_route = pd.DataFrame(response['response']['routes'])
            df_route.columns = ['route2', 'route2_name', 'route2_description']
            df_route['category'] = cat
            df_route['route1'  ] = route
            list_df_routes.append(df_route)
        except:
            pass

    try:
        df_cat = pd.concat(list_df_routes)
        df_cat = df_cat.reset_index(drop=True)
        df_cat = df_cat.set_index(['category', 'route1']).reset_index()
        list_df_cat.append(df_cat)
    except:
        pass
    print('')

df_loop = pd.concat(list_df_cat)

display(df_loop.head(), df_loop.tail())

In [ ]:
df_routes2 = df_routes.merge(df_loop, how = 'left', on = ['category', 'route1'])
df_routes2['ROUTE_FULL'] = df_routes2['route1'] + '/' + df_routes2['route2']
df_routes2.loc[df_routes2['ROUTE_FULL'].isna(), 'ROUTE_FULL'] = df_routes2['route1']
df_routes2

***

Facets

***

In [ ]:
# categories = df_routes2[~df_routes2['route2'].isna()]
categories = df_routes2['category'].unique()
categories

In [ ]:
start_time = time.time()


root_ = 'https://api.eia.gov/v2'

df_loop = df_routes2.copy()

list_df_cats = []
for cat in categories:
    print(''); print(cat); print('')
    
    cat_ = f'/{cat}'
    
    routes = df_loop[df_loop['category'] == cat]['ROUTE_FULL'].unique()
    print(routes)
    
    list_df_routes = []
    
    for route in tqdm(routes):

        try:
            route_ = f'/{route}/facet'
            api_key_ = f'/?api_key={api_key}'
            
            query = f"{root_}{cat_}{route_}{api_key_}"
                            
            response = requests.get(query).text
            response = response.replace('null', '"null"')
            response = ast.literal_eval(response)
            
            facets = response['response']['facetOptions']
            df_facets = pd.DataFrame(facets, columns = ['facetOption'])
            df_facets['category'] = cat
            df_facets['ROUTE_FULL'] = route
            list_df_routes.append(df_facets)
        except:
            pass
    try:        
        df_route_facets = pd.concat(list_df_routes)
        df_route_facets = df_route_facets.reset_index(drop = True)
        list_df_cats.append(df_route_facets)
    except:
        pass
        
df_loop = pd.concat(list_df_cats)
df_loop = df_loop.reset_index(drop = True)
df_loop = df_loop.set_index(['category', 'ROUTE_FULL']).reset_index()
# df_loop = df_loop.drop(['route2_name', 'route2_description'], axis = 1)
# df_loop = df_loop.dropna()


print(''); print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---"); print('')

display(df_loop.head(), df_loop.tail())

In [ ]:
df_routes3 = df_routes2.merge(df_loop, how = 'left', on = ['category', 'ROUTE_FULL'])
df_routes3

In [ ]:
start_time = time.time()


root_ = 'https://api.eia.gov/v2'

df_loop = df_routes3.copy()
categories = list(df_loop['category'].unique())

list_df_cat = []

for cat in categories:
    print('');print(''); print(f'Category: {cat}'); print('');print('')
    
    df = df_loop[df_loop['category'] == cat]
    routes = df['ROUTE_FULL'].unique()
    print(f'Routes available: {routes}')
    print('')

    list_df_routes = []

    for route in routes:
        print(''); print(f'Route: {route}'); print('')
        df = df_loop[df_loop['category'] == cat]
        df = df[df['ROUTE_FULL'] == route]
        facets = df['facetOption'].unique()
        print(f'Facets: {facets}')

        list_df_facets = []

        for facet in tqdm(facets):
            try:
                route_ = f'/{cat}/{route}/facet/{facet}'
                api_key_ = f'/?api_key={api_key}'
                
                query = f"{root_}{route_}{api_key_}"
                                
                response = requests.get(query).text
                response = response.replace('null', '"null"')
                response = ast.literal_eval(response)
                
                df_facet = pd.DataFrame(response['response']['facets'])
                df_facet['category'] = cat
                df_facet['ROUTE_FULL'] = route
                df_facet['facetOption'] = facet
                list_df_facets.append(df_facet)
            except:
                pass
        try:
            df_facets = pd.concat(list_df_facets)
            df_facets = df_facets.reset_index(drop=True)
            list_df_routes.append(df_facets)
        except:
            pass
    try:
        df_cat = pd.concat(list_df_routes)
        df_cat = df_cat.reset_index(drop=True)
        list_df_cat.append(df_cat)
    except:
        pass
            
df_loop = pd.concat(list_df_cat)
df_loop = df_loop.drop_duplicates()
df_loop = df_loop.reset_index(drop = True)
df_loop = df_loop.set_index(['category', 'ROUTE_FULL', 'facetOption']).reset_index()


print(''); print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---");print('')


display(df_loop.head(), df_loop.tail())


In [ ]:
df_routes4 = df_routes3.merge(df_loop, how = 'left', on = ['category', 'ROUTE_FULL', 'facetOption'])
df_routes4 = df_routes4.rename(columns = {'id':'facet', 'name':'facet_name'})
df_routes4['category_name'] = df_routes4['category'].str.title()
df_routes4 = df_routes4.set_index(['category_name', 'ROUTE_FULL']).reset_index()
df_routes4 = df_routes4.drop_duplicates()
df_routes4 = df_routes4.reset_index(drop=True)
display(df_routes4.head(), df_routes4.tail())

In [ ]:
# df_routes4.to_excel(os.path.join(path_config, 'df_routes4.xlsx'))

***

Data Types

***

In [ ]:
root_ = 'https://api.eia.gov/v2'

categories = list(df_routes4['category'].unique())

list_df_cat = []

for cat in categories:

    print(''); print(cat)
    df_loop = df_routes4.copy()
    df_loop = df_loop[df_loop['category'] == cat]
    routes = list(df_loop['ROUTE_FULL'].unique())
    
    list_df_data = []

    for route in tqdm(routes):
        cat_ = f'/{cat}'
        route_ = f'/{route}'
        api_key_ = f'/?api_key={api_key}'
        
        query = f"{root_}{cat_}{route_}{api_key_}"
                            
        try:
            response = requests.get(query).text
            response = response.replace('null', '"null"')
            response = ast.literal_eval(response)
            
            response = response['response']['data']
            response = {k: v for k, v in response.items() if v}
            
            df_data = pd.DataFrame(response).T.reset_index(names = 'data_type')
            df_data['ROUTE_FULL'] = route
            df_data['category'] = cat
            if 'unit' in df_data.columns:
                df_data = df_data.rename(columns = {'unit':'units'})
            
            list_df_data.append(df_data)
        
        except:
            pass

    df_cat = pd.concat(list_df_data)
    df_cat = df_cat.reset_index(drop=True)
    list_df_cat.append(df_cat)

df_data_types = pd.concat(list_df_cat)
df_data_types = df_data_types.reset_index(drop=True)
df_data_types = df_data_types.set_index('category').reset_index()

df_data_types.head()


In [ ]:

df_api = df_routes4.merge(df_data_types, how = 'left', on = ['category', 'ROUTE_FULL'])
df_api = df_api.drop_duplicates()
df_api = df_api.reset_index(drop = True)
df_api = df_api.rename(columns = {'alias_x':'facet_alias', 'alias_y':'data_type_alias'})

display(df_api)


***

Exporting

***

In [ ]:


path_sdl = r'I:/Projects/Josh/Regional Monitoring'
# df_api.to_excel(os.path.join(path_sdl, 'All Routes and Facets FINAL.xlsx'), index=False)


# Might just be to replace this workbook, honestly i forget
# df_api.to_excel(os.path.join(path_sdl, 'Routes.xlsx'), index=False)???

